In [1]:
!pip install pandas scikit-learn transformers torch google-generativeai fastapi uvicorn nest-asyncio pyngrok -q

In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json

data = {
    "text": [
        "نباتي عنده بقع صفراء على الأوراق",
        "الأوراق بتتعفن وبتتساقط",
        "في حشرات على النبات بتاعي",
        "النبات بتاعي مش بينمو صح",
        "شايف بقع بنية على الثمار",
        "الأوراق اتلونت باللون الأصفر",
        "النبات عنده مرض إيه",
        "عايز أعرف المرض اللي في نباتي",
        "my plant has yellow spots on leaves",
        "the leaves are rotting and falling",
        "there are insects on my plant",
        "my plant is not growing properly",
        "brown spots on the fruits",
        "leaves turned yellow",
        "what disease does my plant have",
        "help me identify plant disease",
        "عايز أعرف جودة المحصول بتاعي",
        "المحصول ده Grade A ولا B",
        "إزاي أصنف المحصول",
        "قيّم جودة الطماطم دي",
        "المحصول بتاعي صالح للتصدير",
        "عايز أعرف تقييم المنتج",
        "what is the quality of my crop",
        "is this Grade A or B",
        "how to classify my harvest",
        "evaluate the quality of these tomatoes",
        "is my crop suitable for export",
        "المحصول ده ينفع يتصدر لأوروبا",
        "إيه معايير تصدير الخليج",
        "عايز أصدر منتجاتي برا",
        "هل المحصول مطابق لمعايير التصدير",
        "إيه الأسواق المناسبة للتصدير",
        "معايير الاتحاد الأوروبي للخضار",
        "can I export this crop to Europe",
        "what are Gulf export standards",
        "I want to export my products abroad",
        "does my crop meet export requirements",
        "what markets are suitable for export",
        "درجة الحرارة عالية أوي النهارده",
        "الرطوبة قليلة في الحقل",
        "رطوبة التربة مناسبة للزراعة",
        "إيه الظروف المناسبة للنبات",
        "التربة محتاجة ري",
        "الجو مناسب للزراعة النهارده",
        "temperature is too high today",
        "humidity is low in the field",
        "soil moisture is good for farming",
        "what are the ideal conditions for plants",
        "soil needs irrigation",
        "مرحبا",
        "إزيك",
        "عايز مساعدة",
        "شكراً",
        "ممكن تساعدني",
        "hello",
        "hi there",
        "I need help",
        "thank you",
        "can you help me",
    ],
    "intent": [
        "disease_detection", "disease_detection", "disease_detection", "disease_detection",
        "disease_detection", "disease_detection", "disease_detection", "disease_detection",
        "disease_detection", "disease_detection", "disease_detection", "disease_detection",
        "disease_detection", "disease_detection", "disease_detection", "disease_detection",
        "quality_assessment", "quality_assessment", "quality_assessment",
        "quality_assessment", "quality_assessment", "quality_assessment",
        "quality_assessment", "quality_assessment", "quality_assessment",
        "quality_assessment", "quality_assessment",
        "export_intelligence", "export_intelligence", "export_intelligence",
        "export_intelligence", "export_intelligence", "export_intelligence",
        "export_intelligence", "export_intelligence", "export_intelligence",
        "export_intelligence", "export_intelligence",
        "environment_analysis", "environment_analysis", "environment_analysis",
        "environment_analysis", "environment_analysis", "environment_analysis",
        "environment_analysis", "environment_analysis", "environment_analysis",
        "environment_analysis", "environment_analysis",
        "general", "general", "general", "general", "general",
        "general", "general", "general", "general", "general",
    ]
}

df = pd.DataFrame(data)
print(f"✅ Dataset created: {len(df)} samples")
print(df['intent'].value_counts())

✅ Dataset created: 59 samples
intent
disease_detection       16
quality_assessment      11
export_intelligence     11
environment_analysis    11
general                 10
Name: count, dtype: int64


In [19]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = "aubmindlab/bert-base-arabertv2"
print("⏳ Loading AraBERT...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
arabert_model = AutoModel.from_pretrained(MODEL_NAME)
arabert_model.eval()
print("✅ AraBERT loaded!")

def get_embedding(text):
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128, padding=True
    )
    with torch.no_grad():
        outputs = arabert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

print("⏳ Generating embeddings...")
embeddings = np.array([get_embedding(text) for text in df['text']])
print(f"✅ Embeddings shape: {embeddings.shape}")

⏳ Loading AraBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ AraBERT loaded!
⏳ Generating embeddings...
✅ Embeddings shape: (59, 768)


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels = le.fit_transform(df['intent'])
print("✅ Intent classes:", le.classes_)

def predict_intent(user_text, threshold=0.6):
    user_embedding = get_embedding(user_text).reshape(1, -1)
    similarities = cosine_similarity(user_embedding, embeddings)[0]
    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]
    top_intents = df['intent'].iloc[top_indices].values
    top_texts = df['text'].iloc[top_indices].values
    best_score = top_scores[0]
    best_intent = top_intents[0]
    if best_score < threshold:
        best_intent = "general"
    return {
        "intent": best_intent,
        "confidence": float(best_score),
        "top_matches": [
            {"text": top_texts[i], "intent": top_intents[i], "score": float(top_scores[i])}
            for i in range(len(top_indices))
        ]
    }

print("✅ Intent Classifier ready!")

✅ Intent classes: ['disease_detection' 'environment_analysis' 'export_intelligence'
 'general' 'quality_assessment']
✅ Intent Classifier ready!


In [21]:
knowledge_base = {
    "disease_detection": [
        {
            "topic": "البقع الصفراء على الأوراق",
            "content_ar": "البقع الصفراء على الأوراق غالباً بتكون علامة على نقص المغنيسيوم أو الحديد، أو مرض الاصفرار الفيروسي. العلاج: رش محلول كبريتات المغنيسيوم أو استخدام مبيد فطري مناسب.",
            "content_en": "Yellow spots on leaves are usually a sign of magnesium or iron deficiency, or viral yellowing disease. Treatment: spray magnesium sulfate solution or use appropriate fungicide.",
        },
        {
            "topic": "البقع البنية على الثمار",
            "content_ar": "البقع البنية على الثمار علامة على مرض اللفحة أو التعفن. العلاج: إزالة الثمار المصابة فوراً ورش مبيد فطري نحاسي.",
            "content_en": "Brown spots on fruits indicate blight or rot disease. Treatment: remove infected fruits immediately and spray copper-based fungicide.",
        },
        {
            "topic": "الحشرات على النبات",
            "content_ar": "وجود حشرات على النبات يستدعي تحديد نوع الحشرة أولاً. المن والذباب الأبيض من أكثر الحشرات شيوعاً. العلاج: رش مبيد حشري مناسب أو استخدام المكافحة الحيوية.",
            "content_en": "Insects on plants require identifying the type first. Aphids and whiteflies are most common. Treatment: spray appropriate insecticide or use biological control.",
        },
    ],
    "quality_assessment": [
        {
            "topic": "معايير Grade A",
            "content_ar": "Grade A: محصول ممتاز بدون عيوب، حجم موحد، لون مثالي. مناسب للتصدير لأوروبا والخليج وبيحقق أعلى سعر في السوق.",
            "content_en": "Grade A: excellent crop without defects, uniform size, perfect color. Suitable for export to Europe and Gulf, achieves highest market price.",
        },
        {
            "topic": "معايير Grade B",
            "content_ar": "Grade B: محصول جيد مع عيوب بسيطة في الشكل أو اللون. مناسب للسوق المحلي والتصنيع الغذائي.",
            "content_en": "Grade B: good crop with minor shape or color defects. Suitable for local market and food processing.",
        },
        {
            "topic": "معايير Grade C",
            "content_ar": "Grade C: محصول مقبول مع عيوب واضحة. يُستخدم في التصنيع أو العلف الحيواني. غير مناسب للتصدير.",
            "content_en": "Grade C: acceptable crop with visible defects. Used in processing or animal feed. Not suitable for export.",
        },
    ],
    "export_intelligence": [
        {
            "topic": "معايير تصدير الاتحاد الأوروبي",
            "content_ar": "الاتحاد الأوروبي يشترط: شهادة صحية، خلو من المبيدات المحظورة، Grade A فقط، تغليف معياري. أعلى سعر لكن أصعب متطلبات.",
            "content_en": "EU requires: health certificate, free from banned pesticides, Grade A only, standard packaging. Highest price but strictest requirements.",
        },
        {
            "topic": "معايير تصدير الخليج",
            "content_ar": "دول الخليج تقبل Grade A وB، تشترط شهادة حلال، شهادة منشأ، خلو من الآفات. سوق كبير ومتنامي للمنتجات المصرية.",
            "content_en": "Gulf countries accept Grade A and B, require halal certificate, certificate of origin, pest-free. Large and growing market for Egyptian products.",
        },
        {
            "topic": "السوق المحلي",
            "content_ar": "السوق المحلي يقبل كل الدرجات A وB وC. متطلبات أقل لكن سعر أقل. مناسب للمحاصيل التي لا تستوفي معايير التصدير.",
            "content_en": "Local market accepts all grades A, B and C. Less requirements but lower price. Suitable for crops that don't meet export standards.",
        },
    ],
    "environment_analysis": [
        {
            "topic": "درجة الحرارة المثالية",
            "content_ar": "معظم المحاصيل تحتاج 20-30 درجة مئوية. فوق 35 درجة بيسبب إجهاد حراري للنبات. تحت 10 درجات بيوقف النمو.",
            "content_en": "Most crops need 20-30°C. Above 35°C causes heat stress. Below 10°C stops growth.",
        },
        {
            "topic": "الرطوبة المثالية",
            "content_ar": "الرطوبة المثالية 50-70%. رطوبة عالية فوق 80% بتشجع الأمراض الفطرية. رطوبة منخفضة تحت 40% بتسبب جفاف النبات.",
            "content_en": "Ideal humidity 50-70%. High humidity above 80% encourages fungal diseases. Low humidity below 40% causes plant dehydration.",
        },
        {
            "topic": "رطوبة التربة",
            "content_ar": "رطوبة التربة المثالية 40-60%. أقل من 30% النبات محتاج ري فوري. أكثر من 70% بيسبب تعفن الجذور.",
            "content_en": "Ideal soil moisture 40-60%. Below 30% plant needs immediate irrigation. Above 70% causes root rot.",
        },
    ],
    "general": [
        {
            "topic": "ترحيب",
            "content_ar": """أهلاً وسهلاً بك في نظام الزراعة الذكية! 🌱

أنا مساعدك الزراعي الذكي، وأقدر أساعدك في:

🌿 اكتشاف أمراض النباتات
   ← روح لقسم 'اكتشاف الأمراض' وارفع صورة النبات

⭐ تقييم جودة المحصول
   ← روح لقسم 'تقييم الجودة' وارفع صورة المحصول

🚢 ذكاء التصدير
   ← روح لقسم 'التصدير' وادخل بيانات المحصول

🌡️ تحليل البيئة الزراعية
   ← روح لقسم 'البيئة الذكية' وادخل درجة الحرارة والرطوبة ورطوبة التربة

💬 أو كلمني هنا مباشرة وأنا هساعدك!""",
            "content_en": """Welcome to the Smart Agriculture System! 🌱

I'm your AI agricultural assistant. Here's what I can help you with:

🌿 Plant Disease Detection
   ← Go to 'Disease Detection' and upload a plant photo

⭐ Crop Quality Assessment
   ← Go to 'Quality Assessment' and upload a crop photo

🚢 Export Intelligence
   ← Go to 'Export' and enter your crop details

🌡️ Smart Environment Analysis
   ← Go to 'Smart Environment' and enter temperature, humidity and soil moisture

💬 Or just chat with me here and I'll guide you!""",
        },
        {
            "topic": "مساعدة عامة",
            "content_ar": """أقدر أساعدك في أي من الخدمات دي:

🌿 عندك نبات مريض؟ ← قولي الأعراض أو ارفع صورة في قسم اكتشاف الأمراض
⭐ عايز تعرف جودة محصولك؟ ← ارفع صورة في قسم تقييم الجودة
🚢 عايز تصدر؟ ← ادخل بيانات المحصول في قسم التصدير
🌡️ عايز تحلل بيئة حقلك؟ ← ادخل القراءات في قسم البيئة الذكية""",
            "content_en": """I can help you with any of these services:

🌿 Sick plant? ← Tell me symptoms or upload photo in Disease Detection
⭐ Want to know crop quality? ← Upload photo in Quality Assessment
🚢 Want to export? ← Enter crop details in Export section
🌡️ Want to analyze environment? ← Enter readings in Smart Environment""",
        },
    ],
}

def retrieve_context(intent, lang='ar', top_k=2):
    docs = knowledge_base.get(intent, knowledge_base['general'])
    context = []
    for doc in docs[:top_k]:
        content = doc['content_ar'] if lang == 'ar' else doc['content_en']
        context.append(f"- {doc['topic']}: {content}")
    return "\n".join(context)

print("✅ Knowledge Base ready!")

✅ Knowledge Base ready!


In [28]:
import google.generativeai as genai

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Loaded from Secrets")
except:
    GEMINI_API_KEY = input("Enter Gemini API Key: ")

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")
print("✅ Gemini ready!")

def detect_language(text):
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return 'ar' if arabic_chars > len(text) * 0.3 else 'en'

def build_prompt(user_text, intent, context, lang='ar'):
    if lang == 'ar':
        system = """أنت مساعد زراعي ذكي. اجب بالعربية بشكل محدد وعملي ولا تتجاوز 150 كلمة."""
    else:
        system = """You are a smart agricultural assistant. Answer in English, be specific and practical, max 150 words."""
    return f"""{system}

المعلومات المتاحة:
{context}

سؤال المستخدم: {user_text}

الإجابة:"""

def llm_client(user_text, intent, lang='ar'):
    context = retrieve_context(intent, lang)
    prompt = build_prompt(user_text, intent, context, lang)
    response = gemini_model.generate_content(prompt)
    return response.text

print("✅ LLM Client ready!")

Enter Gemini API Key: AQ.Ab8RN6JYc7P7xjqR3W1rpi8Yt4lIGPy8JXRAuYyMrQFjiDEyIg
✅ Gemini ready!
✅ LLM Client ready!


In [29]:
import time

def chatbot_pipeline(user_text):
    lang = detect_language(user_text)
    intent_result = predict_intent(user_text)
    intent = intent_result['intent']
    confidence = intent_result['confidence']
    response = llm_client(user_text, intent, lang)
    return {
        "user_text": user_text,
        "language": lang,
        "intent": intent,
        "confidence": round(confidence, 3),
        "response": response
    }

print("✅ Pipeline ready!")

# Quick test
result = chatbot_pipeline("مرحبا")
print(f"🤖 Test: {result['response'][:100]}...")

✅ Pipeline ready!


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4669.01ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2055.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2709.79ms


🤖 Test: أهلاً وسهلاً بك في نظام الزراعة الذكية! 🌱 أنا مساعدك الزراعي الذكي.

كيف أقدر أساعدك اليوم؟ أنا هنا ...


In [30]:
import pickle
import os

os.makedirs('/content/models', exist_ok=True)

intent_classifier_data = {
    "embeddings": embeddings,
    "intents": df['intent'].values,
    "texts": df['text'].values,
    "label_encoder": le,
    "model_name": MODEL_NAME,
    "threshold": 0.6
}

with open('/content/models/intent_classifier.pkl', 'wb') as f:
    pickle.dump(intent_classifier_data, f)

with open('/content/models/knowledge_base.pkl', 'wb') as f:
    pickle.dump(knowledge_base, f)

config = {
    "gemini_model": "gemini-2.5-flash",
    "arabert_model": MODEL_NAME,
    "threshold": 0.6,
    "supported_intents": list(le.classes_),
    "supported_languages": ["ar", "en"]
}

with open('/content/models/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("✅ Models saved!")
for f in os.listdir('/content/models'):
    size = os.path.getsize(f'/content/models/{f}') / 1024
    print(f"  - {f} ({size:.1f} KB)")

✅ Models saved!
  - knowledge_base.pkl (6.7 KB)
  - intent_classifier.pkl (179.9 KB)
  - config.pkl (0.3 KB)


In [31]:
import pickle

# Load models
with open('/content/models/intent_classifier.pkl', 'rb') as f:
    classifier_data = pickle.load(f)

with open('/content/models/knowledge_base.pkl', 'rb') as f:
    knowledge_base = pickle.load(f)

print("✅ Models loaded for API!")

✅ Models loaded for API!


In [33]:
import nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import pickle
import time
import threading

nest_asyncio.apply()

# ===== Load Models =====
with open('/content/models/intent_classifier.pkl', 'rb') as f:
    classifier_data = pickle.load(f)

with open('/content/models/knowledge_base.pkl', 'rb') as f:
    knowledge_base = pickle.load(f)

print("✅ Models loaded!")

# ===== FastAPI App =====
app = FastAPI(title="Smart Agriculture Chatbot API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
    allow_credentials=True,
    expose_headers=["*"],
)

@app.middleware("http")
async def add_cors_headers(request, call_next):
    response = await call_next(request)
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Methods"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "*"
    return response

# ===== Request/Response Models =====
class ChatRequest(BaseModel):
    message: str

class ChatResponse(BaseModel):
    message: str
    intent: str
    confidence: float
    language: str

# ===== Helper Functions =====
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )
    with torch.no_grad():
        outputs = arabert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

def detect_language(text):
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return 'ar' if arabic_chars > len(text) * 0.3 else 'en'

def predict_intent(user_text, threshold=0.6):
    user_embedding = get_embedding(user_text).reshape(1, -1)
    similarities = cosine_similarity(user_embedding, classifier_data['embeddings'])[0]
    top_idx = similarities.argmax()
    top_score = similarities[top_idx]
    top_intent = classifier_data['intents'][top_idx]
    if top_score < threshold:
        top_intent = "general"
    return top_intent, float(top_score)

def retrieve_context(intent, lang):
    docs = knowledge_base.get(intent, knowledge_base['general'])
    context = []
    for doc in docs[:2]:
        content = doc['content_ar'] if lang == 'ar' else doc['content_en']
        context.append(f"- {doc['topic']}: {content}")
    return "\n".join(context)

def generate_response(user_text, intent, lang):
    context = retrieve_context(intent, lang)
    if lang == 'ar':
        system = """أنت مساعد زراعي ذكي. اجب بالعربية بشكل محدد وعملي ولا تتجاوز 150 كلمة."""
    else:
        system = """You are a smart agricultural assistant. Answer in English, be specific and practical, max 150 words."""

    prompt = f"""{system}

المعلومات المتاحة:
{context}

سؤال المستخدم: {user_text}

الإجابة:"""

    result = [None]

    def call_gemini():
        try:
            response = gemini_model.generate_content(prompt)
            result[0] = response.text
        except Exception:
            pass

    thread = threading.Thread(target=call_gemini)
    thread.start()
    thread.join(timeout=15)

    if result[0]:
        return result[0]

    # Fallback فوري من الـ Knowledge Base
    docs = knowledge_base.get(intent, knowledge_base['general'])
    if docs:
        content = docs[0]['content_ar'] if lang == 'ar' else docs[0]['content_en']
        return content

    return "عذرًا، حدث خطأ مؤقت. حاول مرة أخرى." if lang == 'ar' else "Sorry, temporary error. Please try again."

# ===== Endpoints =====
@app.get("/")
def root():
    return {"status": "Smart Agriculture Chatbot API is running 🌱"}

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    lang = detect_language(request.message)
    intent, confidence = predict_intent(request.message)
    response = generate_response(request.message, intent, lang)
    return ChatResponse(
        message=response,
        intent=intent,
        confidence=round(confidence, 3),
        language=lang
    )

@app.get("/health")
def health():
    return {"status": "healthy", "model": "gemini-2.5-flash"}

# ===== Run Server =====
ngrok.set_auth_token("3FYWfb4FQRQMOPanzCrkqxt2Cwe_27MtxXgszg1dWQvRgBzAe")
public_url = ngrok.connect(8000)
print(f"🌍 Public URL: {public_url}")
print(f"📡 API Docs: {public_url}/docs")
print("✅ Server is running!")

config_uvicorn = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config_uvicorn)
await server.serve()

✅ Models loaded!
🌍 Public URL: NgrokTunnel: "https://stash-collapse-snowbound.ngrok-free.dev" -> "http://localhost:8000"
📡 API Docs: NgrokTunnel: "https://stash-collapse-snowbound.ngrok-free.dev" -> "http://localhost:8000"/docs
✅ Server is running!


INFO:     Started server process [3165]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     41.47.82.46:0 - "OPTIONS /chat HTTP/1.1" 200 OK


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2007.94ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1571.24ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2001.67ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1220.31ms


INFO:     41.47.82.46:0 - "POST /chat HTTP/1.1" 200 OK


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8468.52ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1625.83ms
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3165]
